# 01 — Exploration des données (caractéristiques mensuelles)

**Objectif :** comprendre la structure du panel `chars_final_with_names.parquet` avant toute modélisation.

**Ce qu'on NE fait PAS ici :** aucun modèle, aucun split train/test, aucune imputation, aucun scaling.
On regarde, on vérifie, on note les pièges.

**Rappel de la structure du problème :**
- C'est un **panel** : une ligne = une action (`permno`) × un mois (`eom`).
- On veut prédire, en **coupe transversale** (*cross-section*), quelles actions feront mieux que les autres le mois suivant.
- La cible `ret_exc_lead1m` est un rendement **futur** : c'est la seule colonne qu'on n'aura jamais le droit d'utiliser comme input.

Le fichier des 8-K (texte) sera exploré dans un notebook séparé.

## 1. Imports

In [ ]:
from pathlib import Path
import datetime as dt

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## 2. Chemins

On calcule les chemins à partir de la racine du projet, pour que le notebook fonctionne
qu'il soit lancé depuis `notebooks/` ou depuis la racine.

⚠️ Le dossier réel s'appelle `data/` (minuscule). On ne modifie jamais son contenu.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

CHARS_PATH = DATA_DIR / "chars_final_with_names.parquet"
FILINGS_PATH = DATA_DIR / "8k_20150101_20260831_identified.parquet"
FEATURE_LIST_PATH = DATA_DIR / "factor_char_list.csv"

print(PROJECT_ROOT)
print(CHARS_PATH.exists(), FILINGS_PATH.exists(), FEATURE_LIST_PATH.exists())

## 3. Lister les fichiers Parquet — sans les charger

Un fichier Parquet contient un **en-tête de métadonnées** : nombre de lignes, schéma (noms + types),
découpage en *row groups*. `pq.ParquetFile` lit uniquement cet en-tête : c'est instantané,
même pour un fichier de 400 Mo.

In [ ]:
for path in sorted(DATA_DIR.glob("*.parquet")):
    metadata = pq.ParquetFile(path).metadata
    size_mb = path.stat().st_size / 1e6
    print(f"{path.name:45s} {size_mb:7.0f} Mo | {metadata.num_rows:>9,} lignes | "
          f"{metadata.num_columns:>3} colonnes | {metadata.num_row_groups} row groups")

## 4. Le schéma du fichier de caractéristiques (toujours sans charger)

Le README indique 198 colonnes = **147 caractéristiques** (les prédicteurs de base, listés dans
`factor_char_list.csv`) + **51 colonnes auxiliaires** (identifiants, dates, rendements, classifications).

On vérifie cette affirmation au lieu de la croire.

In [ ]:
chars_schema = pq.ParquetFile(CHARS_PATH).schema_arrow
schema_table = pd.DataFrame({"column": chars_schema.names,
                             "arrow_type": [str(field.type) for field in chars_schema]})

feature_names = pd.read_csv(FEATURE_LIST_PATH)["variable"].tolist()

print("Nombre de caractéristiques dans le CSV :", len(feature_names))
print("Toutes présentes dans le Parquet ?     ", set(feature_names).issubset(chars_schema.names))
print()
print(schema_table["arrow_type"].value_counts())

In [ ]:
# Les colonnes qui ne sont PAS des caractéristiques : c'est là que se trouvent
# les identifiants, les dates et la cible.
auxiliary_columns = [c for c in chars_schema.names if c not in feature_names]
print(len(auxiliary_columns), "colonnes auxiliaires :")
print(auxiliary_columns)

### Les row groups sont triés par date

Chaque *row group* stocke le min et le max de chaque colonne. Si le fichier est trié par `eom`,
un filtre sur la date permet à PyArrow de **sauter** les row groups inutiles
(*predicate pushdown*) : on ne lit que ce dont on a besoin.

In [ ]:
parquet_file = pq.ParquetFile(CHARS_PATH)
eom_index = chars_schema.get_field_index("eom")

for i in [0, 1, 2, parquet_file.metadata.num_row_groups - 1]:
    row_group = parquet_file.metadata.row_group(i)
    stats = row_group.column(eom_index).statistics
    print(f"row group {i:2d}: eom de {stats.min} à {stats.max} ({row_group.num_rows:,} lignes)")

## 5. Charger UN échantillon : l'année 2019, colonnes utiles seulement

**Pourquoi pas tout le fichier ?** 529 082 lignes × 147 colonnes numériques × 8 octets ≈ 620 Mo en mémoire,
juste pour les features. Pour *explorer*, une année suffit.

**Pourquoi 2019 ?** La compétition évalue les mois de détention de janvier 2021 à août 2026.
Explorer en détail uniquement des données **antérieures** à cette période est une bonne discipline :
cela évite de prendre des décisions (choix de features, transformations) inspirées par la période de test.
C'est une forme subtile de *look-ahead bias* : le *data snooping*.

Deux leviers Parquet :
- `columns=` : on ne lit que certaines colonnes (format colonnaire);
- `filters=` : on ne lit que certaines lignes (grâce aux statistiques des row groups).

In [ ]:
id_columns = ["permno", "permco", "gvkey", "iid", "ticker", "company_name"]
time_columns = ["date", "eom"]
return_columns = ["ret", "ret_exc", "ret_exc_lead1m"]   # ret_exc_lead1m = CIBLE (futur)
context_columns = ["me", "size_grp", "ff49"]

columns_to_load = id_columns + time_columns + return_columns + context_columns + feature_names

chars_2019 = pd.read_parquet(
    CHARS_PATH,
    columns=columns_to_load,
    filters=[("eom", ">=", dt.date(2019, 1, 1)), ("eom", "<=", dt.date(2019, 12, 31))],
)

print("shape :", chars_2019.shape)
print(f"mémoire : {chars_2019.memory_usage(deep=True).sum() / 1e6:.0f} Mo")

### 5a. Types de colonnes

In [ ]:
print(chars_2019.dtypes.value_counts())
chars_2019[id_columns + time_columns + return_columns + context_columns].dtypes

À noter :
- `gvkey` et `iid` sont des **chaînes** (zéros initiaux, valeurs non numériques) : il ne faut jamais les convertir en entiers.
- `date` et `eom` arrivent en type `object` (des `datetime.date` Python). On les convertira en `datetime64` quand on en aura besoin.

In [ ]:
chars_2019.head()

### 5b. Valeurs manquantes

En finance, une valeur manquante ne veut **pas** dire zéro : ratio non défini (dénominateur négatif),
historique insuffisant pour estimer un bêta sur 60 mois, donnée comptable absente…

On regarde la proportion de manquants par caractéristique, triée.

In [ ]:
missing_share = chars_2019[feature_names].isna().mean().sort_values(ascending=False)

print("Caractéristiques sans aucun manquant :", (missing_share == 0).sum())
print("Proportion médiane de manquants       :", round(missing_share.median(), 3))
missing_share.head(20).round(3)

**Astuce Parquet :** les statistiques des row groups contiennent aussi le **nombre de nulls**.
On peut donc obtenir le nombre exact de manquants sur **tout le fichier** sans lire une seule valeur.

In [ ]:
null_counts_full_file = pd.Series(0, index=chars_schema.names)
for i in range(parquet_file.metadata.num_row_groups):
    row_group = parquet_file.metadata.row_group(i)
    for j, name in enumerate(chars_schema.names):
        null_counts_full_file[name] += row_group.column(j).statistics.null_count

missing_share_full_file = null_counts_full_file[feature_names] / parquet_file.metadata.num_rows
comparison = pd.DataFrame({"2019": missing_share, "fichier_complet": missing_share_full_file})
comparison.sort_values("fichier_complet", ascending=False).head(10).round(3)

### 5c. Nombre de valeurs uniques

Utile pour repérer :
- les vraies variables continues;
- les variables **discrètes** déguisées en numériques (scores entiers comme `f_score`);
- les identifiants (autant de valeurs uniques que de titres).

In [ ]:
chars_2019[id_columns + time_columns + context_columns].nunique()

In [ ]:
feature_nunique = chars_2019[feature_names].nunique().sort_values()
feature_nunique.head(10)

### 5d. Statistiques descriptives

Quelques caractéristiques classiques de la littérature :
`ret_12_1` (momentum), `be_me` (value), `market_equity` (taille), `ivol_capm_21d` (volatilité idiosyncratique),
`gp_at` (profitabilité), `at_gr1` (investissement).

In [ ]:
example_features = ["ret_12_1", "be_me", "market_equity", "ivol_capm_21d", "gp_at", "at_gr1"]
chars_2019[example_features].describe(percentiles=[0.01, 0.5, 0.99]).T.round(3)

Observations à garder en tête :
- les **échelles** sont très différentes d'une caractéristique à l'autre → il faudra normaliser;
- les **queues** sont épaisses (comparer le max au 99e centile) → une normalisation par rang
  en coupe transversale (comme dans `penalized_linear_hackathon.py`) est un choix classique et robuste.

## 6. Les colonnes temporelles

Deux dates coexistent :
- `eom` : étiquette de fin de mois calendaire (clé d'alignement mensuelle);
- `date` : dernier jour de cotation observé dans le mois (peut précéder `eom`, p. ex. un week-end).

Pour les sections 6 à 10, on a besoin de **tout l'historique**, mais seulement de quelques colonnes légères.

In [ ]:
panel_keys = pd.read_parquet(
    CHARS_PATH,
    columns=["permno", "gvkey", "iid", "id", "date", "eom", "ret_exc", "ret_exc_lead1m"],
)
panel_keys["date"] = pd.to_datetime(panel_keys["date"])
panel_keys["eom"] = pd.to_datetime(panel_keys["eom"])

print("shape :", panel_keys.shape)
print("date <= eom toujours ?  ", (panel_keys["date"] <= panel_keys["eom"]).all())
print("part des lignes avec date < eom :", round((panel_keys["date"] < panel_keys["eom"]).mean(), 3))

## 7. Les identifiants de titres

| Identifiant | Niveau | Remarque |
|---|---|---|
| `permno` | titre (CRSP) | clé principale recommandée |
| `permco` | entreprise (CRSP) | une entreprise peut avoir plusieurs titres |
| `gvkey` + `iid` | entreprise + émission (Compustat) | chaînes de caractères |
| `ticker` | affichage | change dans le temps, réutilisé : **jamais** comme clé |
| `cik` | émetteur SEC | seulement dans le fichier 8-K |

On vérifie que `(permno, eom)` identifie une ligne de façon unique.

In [ ]:
print("id == permno partout ?            ", (panel_keys["id"] == panel_keys["permno"]).all())
print("(permno, eom) unique ?            ", not panel_keys.duplicated(["permno", "eom"]).any())
print("(gvkey, iid, eom) unique ?        ", not panel_keys.duplicated(["gvkey", "iid", "eom"]).any())
print("titres distincts (permno) :       ", panel_keys["permno"].nunique())

## 8. La cible : `ret_exc_lead1m`

Sur une ligne du mois $t$, `ret_exc_lead1m` est le **rendement excédentaire du mois $t+1$**.
Elle est déjà décalée : il ne faut **pas** la décaler à nouveau.

Logique de prédiction :

$$\underbrace{x_{i,t}}_{\text{caractéristiques connues fin du mois } t} \longrightarrow \underbrace{r^e_{i,t+1}}_{\text{rendement du mois suivant}}$$

**Vérification empirique :** pour chaque titre, `ret_exc_lead1m` au mois $t$ doit être égal à
`ret_exc` au mois $t+1$ (quand les deux mois sont consécutifs).

In [ ]:
panel_keys = panel_keys.sort_values(["permno", "eom"])  # ne jamais supposer que les données sont triées

next_row = panel_keys.groupby("permno")[["eom", "ret_exc"]].shift(-1)
is_next_month = next_row["eom"] == panel_keys["eom"] + pd.offsets.MonthEnd(1)
comparable = is_next_month & panel_keys["ret_exc_lead1m"].notna() & next_row["ret_exc"].notna()

gap = (panel_keys.loc[comparable, "ret_exc_lead1m"] - next_row.loc[comparable, "ret_exc"]).abs()
print("paires comparées :", comparable.sum())
print("écart maximal    :", gap.max())

In [ ]:
target_missing = panel_keys["ret_exc_lead1m"].isna()
last_month = panel_keys["eom"].max()

print("cibles manquantes au total        :", target_missing.sum())
print("dont au dernier mois du fichier   :", (target_missing & (panel_keys["eom"] == last_month)).sum())

Au dernier mois (août 2026), le futur n'existe pas encore : la cible est manquante, et il ne faut
surtout **pas** la remplacer par zéro.

### Le mois cible

Pour découper train / validation / test plus tard, on raisonnera sur le **mois du rendement prédit**
(le mois de détention), pas sur le mois des caractéristiques. On crée juste la colonne ici pour la visualiser.

In [ ]:
panel_keys["target_eom"] = panel_keys["eom"] + pd.offsets.MonthEnd(1)

evaluation_start = pd.Timestamp("2021-01-31")
has_target = panel_keys["ret_exc_lead1m"].notna()

before_eval = has_target & (panel_keys["target_eom"] < evaluation_start)
during_eval = has_target & (panel_keys["target_eom"] >= evaluation_start)
print("lignes avec cible AVANT la période d'évaluation :", before_eval.sum(),
      f"({panel_keys.loc[before_eval, 'target_eom'].nunique()} mois)")
print("lignes avec cible PENDANT la période d'évaluation :", during_eval.sum(),
      f"({panel_keys.loc[during_eval, 'target_eom'].nunique()} mois)")

## 9. Période couverte

In [ ]:
months = panel_keys["eom"].drop_duplicates().sort_values()
expected_months = pd.date_range(months.min(), months.max(), freq="ME")

print("premier mois :", months.min().date())
print("dernier mois :", months.max().date())
print("nombre de mois :", len(months), "| mois manquants :", len(expected_months.difference(months)))

## 10. Nombre d'actions par mois

C'est la taille de la **coupe transversale** : à chaque mois, le modèle classe environ ce nombre d'actions.

In [ ]:
stocks_per_month = panel_keys.groupby("eom").agg(
    all_stocks=("permno", "nunique"),
    stocks_with_target=("ret_exc_lead1m", "count"),
)
stocks_per_month.describe().round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(stocks_per_month.index, stocks_per_month["all_stocks"], lw=2, color="#2a6fdb", label="toutes les actions")
ax.plot(stocks_per_month.index, stocks_per_month["stocks_with_target"], lw=2, color="#e07b39", label="avec cible disponible")
ax.axvline(pd.Timestamp("2020-12-31"), color="grey", ls="--", lw=1)
ax.text(pd.Timestamp("2021-02-28"), ax.get_ylim()[0] + 50, "caractéristiques → cibles 2021+", color="grey")
ax.set_title("Nombre d'actions par mois (eom)")
ax.set_ylabel("nombre d'actions")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

La courbe orange tombe à zéro au dernier mois : c'est normal, le rendement de septembre 2026 n'existe pas encore.
La hausse de 2021-2022 correspond à beaucoup de nouvelles cotations. L'univers change donc dans le temps,
ce qui compte pour la comparaison des performances entre périodes.

## 11. Quelques distributions

### La cible

Les rendements mensuels ont des **queues très épaisses**. Pour que l'histogramme soit lisible,
on coupe l'**affichage** aux 1er et 99e centiles (on ne modifie pas les données).

In [ ]:
target = panel_keys["ret_exc_lead1m"].dropna()
low, high = target.quantile([0.01, 0.99])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(target[(target >= low) & (target <= high)], bins=100, color="#2a6fdb")
ax.set_title("ret_exc_lead1m (affichage entre les centiles 1 et 99)")
ax.set_xlabel("rendement excédentaire mensuel (0.01 = 1 %)")
ax.grid(alpha=0.3)
plt.show()

print(target.describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).round(4))

Le maximum est extrême (plusieurs milliers de %). En OLS, ces quelques observations pèsent énormément
dans la perte quadratique : c'est une question à trancher avant la modélisation (winsorisation de la cible
dans le train ? perte robuste ?).

### Quatre caractéristiques (année 2019)

In [ ]:
features_to_plot = ["ret_12_1", "be_me", "market_equity", "ivol_capm_21d"]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, feature in zip(axes, features_to_plot):
    values = chars_2019[feature].dropna()
    low, high = values.quantile([0.01, 0.99])
    ax.hist(values[(values >= low) & (values <= high)], bins=60, color="#2a6fdb")
    ax.set_title(feature)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

`market_equity` est extrêmement asymétrique (quelques méga-capitalisations) : en niveau, elle écraserait
tout le reste dans une régression. Encore un argument pour une transformation par rang **au sein de chaque mois**.

---

## Ce qu'il faut retenir

1. Panel unique sur `(permno, eom)`, 140 mois, ~3 500 à 4 400 actions par mois.
2. La cible `ret_exc_lead1m` est **vérifiée** comme étant le rendement du mois suivant.
3. Beaucoup de caractéristiques ont des manquants, et leur proportion varie.
4. Échelles hétérogènes et queues épaisses → normalisation nécessaire.

## À toi de jouer (avant la modélisation)

1. Choisis une caractéristique très manquante. Calcule sa proportion de manquants **mois par mois**
   (sur `panel_keys` + une colonne de plus). Est-elle stable dans le temps ?
2. La normalisation par rang du script fourni est calculée **mois par mois**. Est-ce une source de leakage ?
   Pourquoi est-ce différent d'un `StandardScaler` ajusté sur tout l'échantillon ?
3. Avec ~72 mois de cibles avant 2021, comment découperais-tu train et validation pour la première fenêtre ?